# Activation Geometry

Clone the repo into Kaggle, validate that the locally generated Gemini-backed prompt set and split manifests are present, then measure last-token residual activations on the discovery split.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
except ImportError:
    UserSecretsClient = None


def read_secret(name: str) -> str:
    if UserSecretsClient is not None:
        try:
            return UserSecretsClient().get_secret(name).strip()
        except Exception:
            pass
    return os.environ.get(name, "").strip()


REPO_URL = read_secret("FRS_REPO_URL")
HF_TOKEN = read_secret("HF_TOKEN")
REPO_DIR = Path("/kaggle/working/false-refusal-suppression")

if not REPO_URL:
    raise RuntimeError("Set FRS_REPO_URL in Kaggle Secrets before running this notebook.")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[train,dev]"], check=True)

os.chdir(REPO_DIR)
src_path = REPO_DIR / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(REPO_DIR)
print("HF token loaded:", bool(HF_TOKEN))

In [ ]:
from pathlib import Path

MODEL_ID = "Qwen/Qwen3-4B"
PILOT_PROMPTS = Path("data/processed/prompts/pilot_prompts_gemini.jsonl")
SPLIT_DIR = Path("data/processed/splits/pilot_gemini")
ACTIVATION_ARTIFACT = Path("artifacts/activations/qwen3_gemini_pilot_discovery.json")
DIRECTION_ARTIFACT = Path("artifacts/directions/qwen3_gemini_pilot_borderline_vs_easy.json")

required_paths = [PILOT_PROMPTS, SPLIT_DIR / "discovery.jsonl"]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing prebuilt local data artifacts. Generate and commit them locally before using Kaggle: " + ", ".join(missing)
    )

print(MODEL_ID)
print(PILOT_PROMPTS)
print(ACTIVATION_ARTIFACT)
print(DIRECTION_ARTIFACT)

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/measure_activations.py",
        "--model-id",
        MODEL_ID,
        "--split-path",
        str(SPLIT_DIR / "discovery.jsonl"),
        "--output",
        str(ACTIVATION_ARTIFACT),
        "--capture-default-modules",
        "--max-module-captures",
        "12",
    ],
    check=True,
 )

subprocess.run(
    [
        sys.executable,
        "scripts/compute_directions.py",
        "--activations",
        str(ACTIVATION_ARTIFACT),
        "--source-group-a",
        "benign_borderline",
        "--source-group-b",
        "benign_easy",
        "--output",
        str(DIRECTION_ARTIFACT),
    ],
    check=True,
 )

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

with open(ACTIVATION_ARTIFACT, "r", encoding="utf-8") as handle:
    activation_artifact = json.load(handle)

with open(DIRECTION_ARTIFACT, "r", encoding="utf-8") as handle:
    direction_artifact = json.load(handle)

ranked_layers = pd.DataFrame(direction_artifact["ranked_layers"])
display(pd.DataFrame([activation_artifact["summary"]]))
display(ranked_layers.head(12))

if not ranked_layers.empty:
    ax = ranked_layers.head(12).plot.bar(
        x="name",
        y="score",
        figsize=(10, 4),
        legend=False,
        title="Top separability scores by layer",
    )
    ax.set_ylabel("separability score")
    plt.xticks(rotation=45, ha="right")
    plt.show()